#### SET UP

In [ ]:
import sqlite3

con = sqlite3.connect("chicago_analysis.db")
%load_ext sql
%sql sqlite:///chicago_analysis.db

## Crime Analysis

#### 1. Which community areas have the most crimes? (Top 10)

In [ ]:
%%sql
WITH area_names AS (
    SELECT DISTINCT community_area_number, community_area_name
    FROM SCHOOLS_DATA
)

SELECT 
    a.community_area_name AS "Community Area",
    PRINTF('%,d', COUNT(c.id)) AS "Number of Crimes"
FROM area_names a
JOIN CRIME_DATA c
    ON a.community_area_number = CAST(c.community_area AS INTEGER)
WHERE c.community_area IS NOT NULL
GROUP BY a.community_area_number
ORDER BY COUNT(c.id) DESC
LIMIT 10

##### Data Quality Investigation Queries*
###### *Refer to README file

In [ ]:
%sql SELECT community_area FROM CRIME_DATA ORDER BY community_area NULLS FIRST LIMIT 100 OFFSET 100

In [ ]:
%sql SELECT COUNT(*) - COUNT(community_area) AS null_count FROM CRIME_DATA

In [ ]:
%sql SELECT community_area, COUNT(*) AS crimes_count FROM CRIME_DATA GROUP BY community_area

In [ ]:

%%sql
SELECT SUM(crime_count) AS grouped_total
FROM (
    SELECT 
        community_area,
        COUNT(*) AS crime_count
    FROM CRIME_DATA
    WHERE community_area IS NOT NULL
    GROUP BY community_area
)

In [ ]:
%%sql

SELECT DISTINCT c.community_area
FROM CRIME_DATA c
LEFT JOIN SCHOOLS_DATA s
    ON CAST(c.community_area AS INTEGER) = s.community_area_number
WHERE s.community_area_number IS NULL
AND c.community_area IS NOT NULL
ORDER BY c.community_area

In [ ]:
%%sql

SELECT COUNT(*) AS unmatched_crimes
FROM CRIME_DATA c
LEFT JOIN SCHOOLS_DATA s
    ON CAST(c.community_area AS INTEGER) = s.community_area_number
WHERE s.community_area_number IS NULL
AND c.community_area IS NOT NULL

#### 2. What are the top 5 most common crime types?

In [ ]:
%%sql 
SELECT primary_type AS crime_type, PRINTF('%,d', COUNT(*)) AS crimes_count 
FROM CRIME_DATA 
GROUP BY primary_type
ORDER BY COUNT(*) DESC
LIMIT 5

#### 3. What % of crimes result in arrest?

In [ ]:
%%sql 
WITH crimes AS (
    SELECT COUNT(*) AS total_crimes,
    FROM CRIME_DATA)
SELECT ROUND(((COUNT(*) * 100.0) / c.total_crimes), 2) AS percentage_crimes_arrests 
FROM crimes c, CRIME_DATA
WHERE arrest = 1

#### 4. Which locations (location_description) are have the most crimes? (Top 5)

In [ ]:
%%sql 
SELECT 
    location_description AS dangerous_locations,
    COUNT(*) AS crime_count
FROM CRIME_DATA 
WHERE location_description IS NOT NULL
GROUP BY location_description
ORDER BY crime_count DESC
LIMIT 5

#### 5. Are crimes more domestic in high hardship areas?

In [ ]:
%%sql 

WITH area_names AS (
    SELECT DISTINCT community_area_number, community_area_name
    FROM SCHOOLS_DATA
)

SELECT 
    a.community_area_name AS "Community Area",
    PRINTF('%,d', COUNT(c.id)) AS "Number of Crimes",
    cd.hardship_index AS "Hardship Index"
FROM area_names a
JOIN CRIME_DATA c
    ON a.community_area_number = CAST(c.community_area AS INTEGER)
JOIN CENSUS_DATA cd 
    ON c.community_area = cd.ca
WHERE c.community_area IS NOT NULL
AND UPPER(c.description) LIKE '%DOMESTIC%'
GROUP BY a.community_area_number
ORDER BY COUNT(c.id) DESC
LIMIT 10

###### I Don't see a direct correlation, here we are showing the top 10 areas with the most crimes and the hardship indexes go up and down, while 60% of them are on the higher side the rest are not, there isn't a set pattern for this hypothesis.

## School Analysis

#### 1. What is the average safety score across all schools?

In [ ]:
%%sql 
SELECT ROUND(AVG(safety_score), 2) AS "Average Safety Score"
FROM SCHOOLS_DATA
WHERE safety_score IS NOT NULL

#### 2. Which school types (Elementary/Middle/High) perform best?

In [ ]:
%sql SELECT MAX(isat_exceeding_math_), MAX(isat_exceeding_reading_) FROM SCHOOLS_DATA


In [ ]:
%%sql
WITH school_stats AS (
    SELECT 
        elementary_or_high_school AS school_type,
        AVG(isat_exceeding_math_) AS math_avg,
        AVG(isat_exceeding_reading_) AS reading_avg
    FROM SCHOOLS_DATA
    GROUP BY elementary_or_high_school
)

SELECT 
    school_type,
    ROUND(math_avg, 2) AS math_isat_average,
    ROUND(reading_avg, 2) AS reading_isat_average,
    ROUND((math_avg + reading_avg) / 2, 2) AS best
FROM school_stats
ORDER BY best DESC

####  3. Top 10 schools by college enrollment

In [ ]:
%%sql
SELECT 
    name_of_school,
    CAST(NULLIF(college_enrollment_rate, 'NDA') AS REAL) AS college_enrollment_rt
FROM SCHOOLS_DATA
ORDER BY college_enrollment_rt DESC
LIMIT 10

#### 4. Schools with safety score below average

In [ ]:
%%sql
WITH safety_stats AS(
    SELECT 
        AVG(safety_score) AS safety_score_avg
    FROM SCHOOLS_DATA
)
SELECT 
    name_of_school, 
    safety_score
FROM SCHOOLS_DATA, safety_stats ss
WHERE safety_score < ss.safety_score_avg
ORDER BY safety_score

#### 5. Is there a pattern between attendance and safety score?

In [ ]:
%%sql 
SELECT 
    average_student_attendance AS min_attendance, 
    safety_score,
    CASE 
        WHEN safety_score >= 70 THEN 'Safe'
        WHEN safety_score >= 50 THEN 'Moderate'
        ELSE 'Unsafe'
    END AS safety_category
FROM SCHOOLS_DATA
WHERE average_student_attendance = (SELECT MIN(average_student_attendance) FROM SCHOOLS_DATA)

In [ ]:
%%sql 
SELECT 
    average_student_attendance AS max_attendance, 
    safety_score,
    CASE 
            WHEN safety_score >= 70 THEN 'Safe'
            WHEN safety_score >= 50 THEN 'Moderate'
            ELSE 'Unsafe'
        END AS safety_category
FROM SCHOOLS_DATA
WHERE average_student_attendance = (SELECT MAX(average_student_attendance) FROM SCHOOLS_DATA)

In [ ]:
%%sql 
SELECT 
    name_of_school,
    average_student_attendance,
    safety_score,
    CASE 
        WHEN safety_score >= 70 THEN 'Safe'
        WHEN safety_score >= 50 THEN 'Moderate'
        ELSE 'Unsafe'
    END AS safety_category
FROM SCHOOLS_DATA
WHERE safety_score IS NOT NULL
ORDER BY safety_score DESC

###### Prior to vizualization data doesnt seem to show any pattern, the lowest attendance school shows a slightly below average safety score, while the highest attendance school shows a even more lower value below the average safety score.

## Cross Table Analysis

#### 1. Do high hardship areas have more crimes?

In [16]:
%%sql

WITH crimes_stats AS(
    SELECT
        COUNT(*) AS crime_count,
        community_area AS community
    FROM CRIME_DATA
    WHERE community_area IS NOT NULL
    GROUP BY community_area
)
SELECT 
    cd.community_area_name, 
    cd.hardship_index,
    cs.crime_count
FROM CENSUS_DATA cd
JOIN crimes_stats cs
ON cd.ca = cs.community
WHERE cd.ca IS NOT NULL
ORDER BY cd.hardship_index DESC
LIMIT 25

 * sqlite:///chicago_analysis.db
Done.


community_area_name,hardship_index,crime_count
Riverdale,98.0,3124
Fuller Park,97.0,2415
South Lawndale,96.0,11827
Englewood,94.0,18433
Gage Park,93.0,6996
West Garfield Park,92.0,12968
New City,91.0,14572
West Englewood,89.0,20830
Washington Park,88.0,7484
North Lawndale,87.0,20417


#### 2. Do high hardship areas have lower school safety scores?


#### 3. Which community areas appear in BOTH top 10 crime AND bottom 10 school safety?

#### 4. Hardship index vs average college enrollment by community area

#### 5. Which community area is the most "at risk" across all 3 metrics?